In [ ]:
import pandas as pd

df = pd.read_parquet("hf://datasets/ButterChicken98/plantvillage-image-text-pairs/data/train-00000-of-00001.parquet")

In [ ]:
df.head()

,image,caption,captions
0,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,[A vibrant green and healthy tomato leaf with ...
1,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato Late blight,[A tomato leaf showing dark brown lesions and ...
2,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,[A vibrant green and healthy tomato leaf with ...
3,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato mosaic virus,[A tomato leaf with mosaic-like patterns of li...
4,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Pepper bell healthy,"[A fresh green bell pepper leaf with a smooth,..."


In [ ]:
df = df.drop(columns=["image"])
print(df.head())
print(df.shape)

               caption                                           captions
0       Tomato healthy  [A vibrant green and healthy tomato leaf with ...
1   Tomato Late blight  [A tomato leaf showing dark brown lesions and ...
2       Tomato healthy  [A vibrant green and healthy tomato leaf with ...
3  Tomato mosaic virus  [A tomato leaf with mosaic-like patterns of li...
4  Pepper bell healthy  [A fresh green bell pepper leaf with a smooth,...
(20638, 2)


In [ ]:
df = df.explode("captions")
print(df.head())
print(df.shape)

              caption                                           captions
0      Tomato healthy  A vibrant green and healthy tomato leaf with s...
0      Tomato healthy  A healthy Solanum lycopersicum leaf, free of d...
0      Tomato healthy  A fresh tomato leaf outdoors, glowing in sunli...
0      Tomato healthy  A clean and healthy tomato leaf image, perfect...
1  Tomato Late blight  A tomato leaf showing dark brown lesions and w...
(82552, 2)


In [ ]:
caption = df["caption"].unique().tolist()
caption, len(caption)

(['Tomato healthy',
  'Tomato Late blight',
  'Tomato mosaic virus',
  'Pepper bell healthy',
  'Potato Early blight',
  'Tomato Early blight',
  'Tomato YellowLeaf Curl Virus',
  'Tomato Target Spot',
  'Pepper bell Bacterial spot',
  'Tomato Septoria leaf spot',
  'Tomato Spider mites Two spotted spider mite',
  'Tomato Bacterial spot',
  'Potato Late blight',
  'Tomato Leaf Mold',
  'Potato healthy'],
 15)

In [ ]:
df.to_csv("dataset.csv", index=False)
df = pd.read_csv("dataset.csv")
df.head()

,caption,captions
0,Tomato healthy,A vibrant green and healthy tomato leaf with s...
1,Tomato healthy,"A healthy Solanum lycopersicum leaf, free of d..."
2,Tomato healthy,"A fresh tomato leaf outdoors, glowing in sunli..."
3,Tomato healthy,"A clean and healthy tomato leaf image, perfect..."
4,Tomato Late blight,A tomato leaf showing dark brown lesions and w...


In [ ]:
num_labels = 15
data_col_name = "captions"
label_col_name = "caption"

In [ ]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
encoder.fit(df[label_col_name].tolist())
df["label"] = encoder.transform(df[label_col_name].tolist())

In [ ]:
from sklearn.model_selection import train_test_split
df_train, df_test = train_test_split(df, train_size=0.8)

In [ ]:
from datasets import Dataset
train_dataset = Dataset.from_pandas(df_train)
test_dataset = Dataset.from_pandas(df_test)
model_name = "distilbert-base-uncased"

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
def tokenize_fn(data):
  return tokenizer(data["captions"], truncation=True)

In [ ]:
tokenized_train = train_dataset.map(tokenize_fn, batched=True)
tokenized_test = test_dataset.map(tokenize_fn, batched=True)

Map:   0%|          | 0/66041 [00:00<?, ? examples/s]

Map:   0%|          | 0/16511 [00:00<?, ? examples/s]

In [ ]:
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels= 15)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00


In [ ]:
import evaluate
from transformers import TrainingArguments, Trainer
import numpy as np

In [ ]:
eval_metrics = evaluate.load("accuracy")

In [ ]:
def metrics(eval_pred):
  logits, labels = eval_pred
  prediction = np.argmax(logits, axis=1)
  return eval_metrics.compute(predictions=prediction, references=labels)

In [ ]:
training_arguments = TrainingArguments(
    output_dir="./checkpoints",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    learning_rate=0.00005,
    save_strategy="epoch",
    logging_strategy="epoch",
    save_total_limit=2,
    weight_decay=0.01,
    report_to="none"

)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_arguments,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics= metrics
)

/tmp/ipython-input-4236874995.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

Step,Training Loss
8256,0.021200
16512,0.000000


TrainOutput(global_step=16512, training_loss=0.010587747333988245, metrics={'train_runtime': 698.1391, 'train_samples_per_second': 189.192, 'train_steps_per_second': 23.651, 'total_flos': 921756740417460.0, 'train_loss': 0.010587747333988245, 'epoch': 2.0})

In [ ]:
results = trainer.evaluate()
print("Final evaluation:", results)

Final evaluation: {'eval_loss': 7.825025960528365e-08, 'eval_accuracy': 1.0, 'eval_runtime': 16.98, 'eval_samples_per_second': 972.38, 'eval_steps_per_second': 121.555, 'epoch': 2.0}


In [ ]:
trainer.save_model("plant_text_classifier")

In [ ]:
pip install torch

In [ ]:
import torch

sample_text = "leaf has brown spots and looks dry"

device = next(model.parameters()).device

inputs = tokenizer(sample_text, return_tensors="pt", truncation=True, padding=True).to(device)

with torch.no_grad():
    outputs = model(**inputs)
    prediction = outputs.logits.argmax(-1).item()

predicted_label = encoder.inverse_transform([[prediction]])[0]
print(f"Sample text: {sample_text}")
print(f"Predicted label: {predicted_label}")


Sample text: leaf has brown spots and looks dry
Predicted label: Tomato Bacterial spot


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:151: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [ ]:
results = trainer.evaluate()
print(results)

{'eval_loss': 7.825025960528365e-08, 'eval_accuracy': 1.0, 'eval_runtime': 21.1641, 'eval_samples_per_second': 780.142, 'eval_steps_per_second': 97.524, 'epoch': 2.0}


In [ ]:
sample_text = "fruit has mold and is soft"
inputs = tokenizer(sample_text, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)
    prediction = outputs.logits.argmax(-1).item()

predicted_label = encoder.inverse_transform([[prediction]])[0]

print("Predicted class:", prediction)
print("Predicted label:", predicted_label)


Predicted class: 8
Predicted label: Tomato Leaf Mold


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:151: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
